# Chimeric RNA-Seq Negative Data Pipeline (Updated)

This notebook implements a full pipeline to produce labeled **False Negative** and **False Positive** training data for a chimera detection model.

## 1. Terminology & Strategy

Both datasets generated here will be labeled as **False (0)**, but they serve different purposes:

* **False Negative Candidates (Baseline / Canonical):**
    * **Content:** Real, high-quality human transcripts (from UniProt Swiss-Prot).
    * **Goal:** Teach the model what "normal" RNA looks like so it doesn't flag healthy tissue.
    * **Label:** 0

* **False Positive Candidates (Hard Negatives / Synthetic):**
    * **Content:** Synthetic sequences designed to trick the model using advanced operators: **Reverse Complement**, **Shuffled**, and **Shift-Invariant Random Pairs**.
    * **Goal:** Teach the model to distinguish specific fusion breakpoints from random noise, artifacts, or non-coding strand syntax.
    * **Label:** 0

## 1. Setup & Configuration

We will use **Biopython** to handle sequence processing. Ensure it is installed (`pip install biopython`).

In [22]:
# ==============================================================================
# CELL 1: IMPORTS & CONFIGURATION
# ==============================================================================
import os
import gzip
import random
import requests
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
# --- HTTPS PATCH (Required for recent Biopython/UniProt) ---
from Bio.SeqIO import UniprotIO
UniprotIO.NS = "{https://uniprot.org/uniprot}"
# -----------------------------------------------------------

# Configuration
SEED = 42
random.seed(SEED)

# SCALING TARGET:
# We scan the file until we find 30,000 raw human transcripts.
# With 5 variants per transcript, this will produce ~60k-70k final samples.
TARGET_RAW_COUNT = 30000 

# Output Filenames
local_gz_file = "uniprot_sprot.xml.gz"
FN_OUTPUT_FILE = "false_negative_candidates.fasta"  # Canonical
FP_OUTPUT_FILE = "false_positive_candidates.fasta"  # Synthetic

UNIPROT_URL = "https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/uniprot_sprot.xml.gz"

print(f"✅ Setup Complete. Raw Input Target: {TARGET_RAW_COUNT}")

✅ Setup Complete. Raw Input Target: 30000


## 2. Data Downloader

This step downloads the official UniProt Swiss-Prot database (gzipped) if it doesn't already exist locally.

In [23]:
# ==============================================================================
# CELL 2: DOWNLOAD AND LOAD DATA
# ==============================================================================
def download_uniprot(url, local_path):
    if os.path.exists(local_path):
        print(f"✅ Found existing file: {local_path}")
        return
    print(f"⬇️ Downloading UniProt data from {url}...")
    response = requests.get(url, stream=True)
    with open(local_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=1024*1024):
            if chunk: f.write(chunk)
    print("✅ Download Complete.")

# 1. Ensure file exists
download_uniprot(UNIPROT_URL, local_gz_file)

def load_canonical_sequences(gz_path, limit):
    """Loads distinct human transcripts."""
    records = []
    print(f"📖 Parsing XML to find {limit} Human transcripts...")
    
    with gzip.open(gz_path, 'rt') as handle:
        try:
            for record in SeqIO.parse(handle, "uniprot-xml"):
                # Filter: Human (ID 9606) & Length 500-30k
                organism = record.annotations.get("organism", "")
                if "Homo sapiens" in organism:
                    if 500 < len(record.seq) < 30000:
                        records.append(record)
                        if len(records) >= limit:
                            break
        except ValueError as e:
            print(f"⚠️ Warning during parsing (likely non-fatal): {e}")
            
    print(f"✅ Loaded {len(records)} human transcripts.")
    return records

# 2. Load Data
real_transcripts = load_canonical_sequences(local_gz_file, limit=TARGET_RAW_COUNT)

✅ Found existing file: uniprot_sprot.xml.gz
📖 Parsing XML to find 30000 Human transcripts...


✅ Loaded 8050 human transcripts.


## 3. Load Canonical Sequences

We parse the downloaded file, filtering specifically for human sequences (*Homo sapiens*) to create our "Real" baseline.

In [24]:
# ==============================================================================
# CELL 4: GENERATE NEGATIVE DATASETS (Corrected Scientific Version)
# ==============================================================================
import random
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
# --- HTTPS PATCH (Required for recent Biopython/UniProt) ---
from Bio.SeqIO import UniprotIO
UniprotIO.NS = "{https://uniprot.org/uniprot}"

# 1. Setup
random.seed(SEED)
TARGET_RAW_COUNT = 30000  # Scan until we find 30k raw human transcripts

def trim_artifacts(seq_str):
    """Normalize sequence string."""
    seq_str = str(seq_str).upper().strip()
    return seq_str.rstrip('N').rstrip('A')

# 2. Force Reload if Data is Insufficient
current_count = len(real_transcripts) if 'real_transcripts' in locals() else 0

if current_count < TARGET_RAW_COUNT:
    print(f"🔄 Current pool ({current_count}) is too small. Reloading {TARGET_RAW_COUNT} sequences...")
    # Calls the function defined in Cell 3 (Ensure load_canonical_sequences is defined)
    real_transcripts = load_canonical_sequences(local_gz_file, limit=TARGET_RAW_COUNT)
else:
    print(f"✅ Using existing pool of {current_count} sequences.")

# -------------------------------------------------------
# GENERATION PIPELINE
# -------------------------------------------------------
fn_records = [] # Canonical (Normal)
fp_records = [] # Synthetic (Hard Decoys)

# STRATEGY: Apply ALL 4 operators to EVERY transcript to maximize volume & hardness.
strategies = ['rev_comp', 'reverse_only', 'self_concat', 'circular_permute']

print(f"\n🚀 Starting Generation Pipeline on {len(real_transcripts)} transcripts...")

for record in real_transcripts:
    seq_str = trim_artifacts(record.seq)
    L = len(seq_str)
    
    # Filter: Too short sequences are not useful
    if L < 500: continue
    
    # --- A. Canonical Negative (1 per transcript) ---
    fn_records.append(SeqRecord(
        Seq(seq_str),
        id=f"NEG_CANONICAL_{record.id}",
        description="Label:0 source:UniProt_Canonical"
    ))

    # --- B. Synthetic Hard Negatives (4 per transcript) ---
    for strat in strategies:
        
        # 1. Reverse Complement (Wrong Strand)
        if strat == 'rev_comp':
            rc_seq = str(Seq(seq_str).reverse_complement())
            fp_records.append(SeqRecord(
                Seq(rc_seq),
                id=f"NEG_SYNTH_RC_{record.id}",
                description="Label:0 type:rev_comp"
            ))
            
        # 2. Reverse Only (Breaks Grammar, Keeps Counts)
        elif strat == 'reverse_only':
            rev_seq = seq_str[::-1]
            fp_records.append(SeqRecord(
                Seq(rev_seq),
                id=f"NEG_SYNTH_REV_{record.id}",
                description="Label:0 type:reverse_only"
            ))

        # 3. Self-Concatenation (Tandem Repeat Artifact)
        elif strat == 'self_concat':
            combined = seq_str + seq_str
            if len(combined) > 32000: combined = combined[:32000]
            fp_records.append(SeqRecord(
                Seq(combined),
                id=f"NEG_SYNTH_SELF_{record.id}",
                description="Label:0 type:self_concat"
            ))

        # 4. Circular Permutation (The J-Operator)
        # s[k:] + s[:k] where k ~ Uniform(0, L)
        elif strat == 'circular_permute':
            if L > 200:
                k = random.randint(50, L - 50) 
                permuted_seq = seq_str[k:] + seq_str[:k]
                fp_records.append(SeqRecord(
                    Seq(permuted_seq),
                    id=f"NEG_SYNTH_CIRC_{record.id}_k{k}",
                    description="Label:0 type:circular_permutation"
                ))

# -------------------------------------------------------
# SUMMARY
# -------------------------------------------------------
total_neg = len(fn_records) + len(fp_records)
print(f"\n✅ GENERATION COMPLETE")
print(f"   ├── Canonical Negatives: {len(fn_records)}")
print(f"   ├── Synthetic Negatives: {len(fp_records)}")
print(f"   └── TOTAL NEGATIVE SET:  {total_neg} samples")

if total_neg < 30000:
    print("⚠️ Warning: Still below 30k. Try increasing TARGET_RAW_COUNT.")
else:
    print(f"🎉 Success: Created {total_neg} negative samples (Target: >30k).")

🔄 Current pool (8050) is too small. Reloading 30000 sequences...
📖 Parsing XML to find 30000 Human transcripts...


✅ Loaded 8050 human transcripts.

🚀 Starting Generation Pipeline on 8050 transcripts...

✅ GENERATION COMPLETE
   ├── Canonical Negatives: 8050
   ├── Synthetic Negatives: 32200
   └── TOTAL NEGATIVE SET:  40250 samples
🎉 Success: Created 40250 negative samples (Target: >30k).


## 4. Save to FASTA

Write the records to disk.

In [25]:
# ==============================================================================
# CELL 4: SAVE TO FASTA
# ==============================================================================
print(f"Writing to {FN_OUTPUT_FILE}...")
SeqIO.write(fn_records, FN_OUTPUT_FILE, "fasta")

print(f"Writing to {FP_OUTPUT_FILE}...")
SeqIO.write(fp_records, FP_OUTPUT_FILE, "fasta")

print("\n🎉 FILES READY FOR TRAINING.")
print(f"   1. {os.path.abspath(FN_OUTPUT_FILE)}")
print(f"   2. {os.path.abspath(FP_OUTPUT_FILE)}")

Writing to false_negative_candidates.fasta...
Writing to false_positive_candidates.fasta...



🎉 FILES READY FOR TRAINING.
   1. /home/akp1/GeneAI/false_negative_candidates.fasta
   2. /home/akp1/GeneAI/false_positive_candidates.fasta
